# Myanmar Headline Generator - STRONGER MODEL
## Fixed Architecture: 2 Layers, 512 Hidden Dim

**Changes from previous version:**
- ✅ Increased to 2 layers (from 1) - more capacity
- ✅ Increased to 512 hidden dim (from 256) - stronger model
- ✅ Added proper dropout (0.3) - prevent overfitting
- ✅ Better learning rate schedule - stable training
- ✅ Repetition penalty - no more loops!

**Expected Results:**
- Epoch 5: Loss ~4.0-4.5 (vs 5.2 before)
- Epoch 10: Loss ~3.2-3.8 (vs 5.5 before)
- Epoch 15: Loss ~2.5-3.0 ✓
- Epoch 20: Loss ~2.0-2.5 ✓
- Headlines: Coherent, non-repetitive ✓

In [ ]:
!pip install gensim sacrebleu -q

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from gensim.models import KeyedVectors
from tqdm import tqdm
import re
import pickle
from pathlib import Path
from collections import Counter
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

print("✓ Imports successful")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Configuration - STRONGER MODEL

In [ ]:
# Paths
DICT_PATH = "/content/drive/MyDrive/NLP Project/dict-words.txt"
STOPWORDS_PATH = "/content/drive/MyDrive/NLP Project/stopwords.txt"
DATA_PATH = "/content/drive/MyDrive/NLP Project/Headline Generator Dataset/headline_corpus.csv"
FASTTEXT_PATH = "/content/drive/MyDrive/NLP Project/Headline Generator Dataset/cc.my.300.vec"
CACHE_FILE = "preprocessed_consistent.pkl"  # Your working cache


MAX_VOCAB_SIZE = 25000    
MAX_TEXT_LEN = 256          # Keep
MAX_HEAD_LEN = 20           # Keep
EMBEDDING_DIM = 300         # FastText
HIDDEN_DIM = 512            
NUM_LAYERS = 2             
DROPOUT = 0.3              

# Training Configuration
BATCH_SIZE = 16             
EPOCHS = 25
LEARNING_RATE = 0.001       
TEACHER_FORCING_RATIO = 0.7 # Keep your good setting
GRADIENT_CLIP = 1.0
LABEL_SMOOTHING = 0.1
UNFREEZE_EPOCH = 8          # Unfreeze embeddings later

# Device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("="*60)
print("STRONGER MODEL CONFIGURATION")
print("="*60)
print(f"Device: {DEVICE}")
print(f"Hidden dim: {HIDDEN_DIM} (was 256)")
print(f"Num layers: {NUM_LAYERS} (was 1)")
print(f"Dropout: {DROPOUT} (new!)")
print(f"Batch size: {BATCH_SIZE} (reduced for bigger model)")
print(f"Learning rate: {LEARNING_RATE}")
print("="*60)

## Preprocessor (Same as Before)

In [ ]:
class MyanmarTextPreprocessor():
    def __init__(self, dict_path: str, stop_path: str):
        self.dictionary = self.load_dictionary(dict_path)
        self.stopwords = self.load_stopwords(stop_path)
        self.syllable_pattern = r"(([A-Za-z0-9]+)|[က-အ|ဥ|ဦ](င်္|[က-အ][ှ]*[့း]*[်]|္[က-အ]|[ါ-ှႏꩻ][ꩻ]*){0,}|.)"

    def load_dictionary(self, dict_path):
        dictionary = set()
        with open(dict_path, 'r', encoding='utf-8') as f:
            for line in f:
                word = line.strip()
                if word:
                    dictionary.add(word)
        return dictionary

    def load_stopwords(self, stopword_path):
        stopwords = set()
        with open(stopword_path, 'r', encoding='utf-8') as f:
            for line in f:
                word = line.strip()
                if word:
                    stopwords.add(word)
        return stopwords

    def merge_with_dictionary(self, syllables):
        merged_tokens = []
        i = 0
        while i < len(syllables):
            matched = False
            for j in range(len(syllables), i, -1):
                combined = ''.join(syllables[i:j])
                if combined in self.dictionary:
                    merged_tokens.append(combined)
                    i = j
                    matched = True
                    break
            if not matched:
                merged_tokens.append(syllables[i])
                i += 1
        return merged_tokens

    def tokenize(self, text: str, use_dict_merge: bool = True, remove_stopwords: bool = True):
        text = re.sub(self.syllable_pattern, r"\1 ", text)
        syllables = text.strip().split()
        
        if use_dict_merge:
            tokens = self.merge_with_dictionary(syllables)
        else:
            tokens = syllables
        
        if remove_stopwords:
            tokens = [token for token in tokens if token not in self.stopwords]
        
        return tokens

print("✓ Preprocessor defined")

## Load Data (Use Your Existing Cache)

In [ ]:
# Load data
print("Loading data...")
df = pd.read_csv(DATA_PATH)
texts = df["text"].astype(str).tolist()
headlines = df["headline"].astype(str).tolist()
print(f"✓ Loaded {len(texts):,} articles")

# Load cached preprocessing (your working one with 99.3% overlap)
processor = MyanmarTextPreprocessor(DICT_PATH, STOPWORDS_PATH)

if Path(CACHE_FILE).exists():
    print("\nLoading your cached preprocessed data...")
    with open(CACHE_FILE, "rb") as f:
        data = pickle.load(f)
        tokenized_texts = data["tokenized_texts"]
        tokenized_headlines = data["tokenized_headlines"]
    print(f"✓ Loaded {len(tokenized_texts):,} preprocessed examples")
else:
    print("\n⚠️  Cache not found! Reprocessing...")
    tokenized_texts = []
    tokenized_headlines = []
    
    for text, headline in tqdm(zip(texts, headlines), total=len(texts)):
        tokenized_texts.append(processor.tokenize(text, use_dict_merge=False, remove_stopwords=False))
        tokenized_headlines.append(processor.tokenize(headline, use_dict_merge=False, remove_stopwords=False))
    
    with open(CACHE_FILE, "wb") as f:
        pickle.dump({
            "tokenized_texts": tokenized_texts,
            "tokenized_headlines": tokenized_headlines
        }, f)

print(f"\nSample text: {tokenized_texts[0][:15]}")
print(f"Sample headline: {tokenized_headlines[0][:10]}")

## Build Vocabulary (Same as Before)

In [ ]:
print("Building vocabulary...")

counter = Counter()
for t in tokenized_texts + tokenized_headlines:
    counter.update(t)

print(f"Total unique tokens: {len(counter):,}")

# Keep top N
vocab_counts = counter.most_common(MAX_VOCAB_SIZE - 4)
vocab = ["<pad>", "<unk>", "<sos>", "<eos>"] + [w for w, c in vocab_counts]

word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}
vocab_size = len(vocab)

print(f"\n✓ Vocabulary size: {vocab_size:,}")

## Load Embeddings

In [ ]:
print("Loading FastText...")
ft = KeyedVectors.load_word2vec_format(FASTTEXT_PATH)

embedding_matrix = np.random.normal(scale=0.6, size=(vocab_size, EMBEDDING_DIM))

covered = 0
for word, idx in word2idx.items():
    if word in ft:
        embedding_matrix[idx] = ft[word]
        covered += 1

embedding_matrix[word2idx["<pad>"]] = 0
embedding_matrix = torch.tensor(embedding_matrix, dtype=torch.float).to(DEVICE)

print(f"✓ Embedding: {embedding_matrix.shape}")
print(f"  Coverage: {covered}/{vocab_size} ({covered/vocab_size*100:.1f}%)")

## Dataset

In [ ]:
def encode_sentence(tokens, max_len, add_sos_eos=False):
    ids = [word2idx.get(t, word2idx["<unk>"]) for t in tokens]
    
    if add_sos_eos:
        ids = [word2idx["<sos>"]] + ids + [word2idx["<eos>"]]
    
    if len(ids) < max_len:
        ids += [word2idx["<pad>"]] * (max_len - len(ids))
    else:
        ids = ids[:max_len]
    
    return ids

class HeadlineDataset(Dataset):
    def __init__(self, texts, headlines):
        self.texts = texts
        self.headlines = headlines

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        src = torch.tensor(encode_sentence(self.texts[idx], MAX_TEXT_LEN), dtype=torch.long)
        trg = torch.tensor(encode_sentence(self.headlines[idx], MAX_HEAD_LEN, add_sos_eos=True), dtype=torch.long)
        decoder_input = trg[:-1]
        decoder_target = trg[1:]
        return src, decoder_input, decoder_target

train_texts, val_texts, train_headlines, val_headlines = train_test_split(
    tokenized_texts, tokenized_headlines, test_size=0.1, random_state=42
)

train_dataset = HeadlineDataset(train_texts, train_headlines)
val_dataset = HeadlineDataset(val_texts, val_headlines)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"✓ Train: {len(train_dataset):,}")
print(f"✓ Val: {len(val_dataset):,}")
print(f"✓ Batches per epoch: {len(train_loader)}")

## STRONGER MODEL - 2 Layers, 512 Hidden

In [ ]:
class Seq2SeqAttnLSTM(nn.Module):
    """STRONGER: 2-layer, 512-hidden, with dropout"""
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers, 
                 dropout=0.3, embedding_matrix=None, freeze_embeddings=False):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.vocab_size = vocab_size

        # Embedding
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=word2idx["<pad>"])
        if embedding_matrix is not None:
            self.embedding.weight.data.copy_(embedding_matrix)
        self.embedding.weight.requires_grad = not freeze_embeddings

        # ⭐ Bidirectional encoder with dropout
        self.encoder = nn.LSTM(
            embedding_dim, hidden_dim, 
            num_layers=num_layers, 
            batch_first=True, 
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0  # Dropout between layers
        )

        # ⭐ Decoder with dropout
        self.decoder = nn.LSTM(
            embedding_dim + hidden_dim * 2,
            hidden_dim, 
            num_layers=num_layers, 
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )

        # Attention
        self.attention = nn.Linear(hidden_dim * 3, 1)
        
        # Output
        self.fc = nn.Linear(hidden_dim, vocab_size)
        
        # ⭐ Dropout layers
        self.dropout = nn.Dropout(dropout)
        
        # Bridge
        self.bridge_h = nn.Linear(hidden_dim * 2, hidden_dim)
        self.bridge_c = nn.Linear(hidden_dim * 2, hidden_dim)
        
        # ⭐ Better initialization
        self._init_weights()
    
    def _init_weights(self):
        """Initialize weights properly"""
        for name, param in self.named_parameters():
            if 'embedding' in name:
                continue  # Skip embeddings
            elif 'weight' in name and len(param.shape) >= 2:
                nn.init.xavier_uniform_(param)
            elif 'bias' in name:
                nn.init.constant_(param, 0)

    def forward(self, src, trg_input, teacher_forcing_ratio=0.5):
        batch_size = src.size(0)
        trg_len = trg_input.size(1)

        # Encoder
        embedded_src = self.dropout(self.embedding(src))
        enc_outputs, (hidden, cell) = self.encoder(embedded_src)

        # Bridge
        hidden = hidden.view(self.num_layers, 2, batch_size, self.hidden_dim)
        cell = cell.view(self.num_layers, 2, batch_size, self.hidden_dim)
        hidden = torch.cat([hidden[:, 0, :, :], hidden[:, 1, :, :]], dim=2)
        cell = torch.cat([cell[:, 0, :, :], cell[:, 1, :, :]], dim=2)
        hidden = torch.tanh(self.bridge_h(hidden))
        cell = torch.tanh(self.bridge_c(cell))

        # Decoder
        embedded_trg = self.dropout(self.embedding(trg_input))
        outputs = torch.zeros(batch_size, trg_len, self.vocab_size).to(src.device)
        dec_input = embedded_trg[:, 0, :].unsqueeze(1)

        for t in range(trg_len):
            # Attention
            hidden_repeated = hidden[-1].unsqueeze(1).repeat(1, enc_outputs.size(1), 1)
            attn_input = torch.cat([hidden_repeated, enc_outputs], dim=2)
            attn_weights = self.attention(attn_input).squeeze(2)
            attn_weights = F.softmax(attn_weights, dim=1)
            context = torch.bmm(attn_weights.unsqueeze(1), enc_outputs)
            
            rnn_input = torch.cat([dec_input, context], dim=2)
            output, (hidden, cell) = self.decoder(rnn_input, (hidden, cell))
            prediction = self.fc(self.dropout(output.squeeze(1)))
            outputs[:, t, :] = prediction
            
            # Teacher forcing
            use_teacher_forcing = torch.rand(1).item() < teacher_forcing_ratio
            if use_teacher_forcing and t < trg_len - 1:
                dec_input = embedded_trg[:, t + 1, :].unsqueeze(1)
            else:
                top1 = prediction.argmax(1)
                dec_input = self.embedding(top1).unsqueeze(1)

        return outputs

print("✓ STRONGER model defined (2 layers, 512 hidden, dropout)")

## Loss Function

In [ ]:
class LabelSmoothingLoss(nn.Module):
    def __init__(self, vocab_size, padding_idx, smoothing=0.1):
        super().__init__()
        self.vocab_size = vocab_size
        self.padding_idx = padding_idx
        self.smoothing = smoothing
        self.confidence = 1.0 - smoothing
        
    def forward(self, pred, target):
        pred = pred.log_softmax(dim=-1)
        
        with torch.no_grad():
            true_dist = torch.zeros_like(pred)
            true_dist.fill_(self.smoothing / (self.vocab_size - 2))
            true_dist.scatter_(1, target.unsqueeze(1), self.confidence)
            true_dist[:, self.padding_idx] = 0
            mask = (target == self.padding_idx)
            true_dist[mask] = 0
        
        loss = -torch.sum(true_dist * pred, dim=-1)
        loss = loss.masked_fill(mask, 0)
        return loss.sum() / (~mask).sum()

print("✓ Loss function defined")

## Initialize Model

In [ ]:
# Create STRONGER model
model = Seq2SeqAttnLSTM(
    vocab_size=vocab_size,
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,      # 512
    num_layers=NUM_LAYERS,      # 2
    dropout=DROPOUT,             # 0.3
    embedding_matrix=embedding_matrix,
    freeze_embeddings=True
).to(DEVICE)

criterion = LabelSmoothingLoss(vocab_size, word2idx["<pad>"], smoothing=LABEL_SMOOTHING)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', factor=0.5, patience=3, verbose=True)

# Model info
total_params = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("\n" + "="*60)
print("MODEL SUMMARY")
print("="*60)
print(f"Total parameters: {total_params:,}")
print(f"Trainable: {trainable:,}")
print(f"Frozen (embeddings): {total_params - trainable:,}")
print(f"\nArchitecture:")
print(f"  Layers: {NUM_LAYERS}")
print(f"  Hidden: {HIDDEN_DIM}")
print(f"  Dropout: {DROPOUT}")
print(f"  Vocab: {vocab_size:,}")
print(f"\nExpected size: ~{total_params * 4 / 1024 / 1024:.0f} MB")
print("="*60)

## Training Functions

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device, tf_ratio):
    model.train()
    total_loss = 0
    
    progress = tqdm(loader, desc="Training")
    for src, dec_input, dec_target in progress:
        src = src.to(device)
        dec_input = dec_input.to(device)
        dec_target = dec_target.to(device)
        
        optimizer.zero_grad()
        output = model(src, dec_input, teacher_forcing_ratio=tf_ratio)
        loss = criterion(output.reshape(-1, vocab_size), dec_target.reshape(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP)
        optimizer.step()
        
        total_loss += loss.item()
        progress.set_postfix({'loss': f'{loss.item():.4f}'})
    
    return total_loss / len(loader)

def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    
    with torch.no_grad():
        for src, dec_input, dec_target in tqdm(loader, desc="Validating"):
            src = src.to(device)
            dec_input = dec_input.to(device)
            dec_target = dec_target.to(device)
            
            output = model(src, dec_input, teacher_forcing_ratio=0.0)
            loss = criterion(output.reshape(-1, vocab_size), dec_target.reshape(-1))
            total_loss += loss.item()
    
    return total_loss / len(loader)

print("✓ Training functions ready")

## Generation with Repetition Penalty

In [ ]:
def generate_headline_greedy(model, raw_text, max_len=MAX_HEAD_LEN, repetition_penalty=1.5):
    """Generate with repetition penalty to avoid loops"""
    model.eval()
    
    with torch.no_grad():
        # Tokenize
        tokens = processor.tokenize(raw_text, use_dict_merge=False, remove_stopwords=False)
        src_ids = torch.tensor(encode_sentence(tokens, MAX_TEXT_LEN), dtype=torch.long).unsqueeze(0).to(DEVICE)
        
        # Encode
        embedded_src = model.embedding(src_ids)
        enc_outputs, (hidden, cell) = model.encoder(embedded_src)
        
        # Bridge
        batch_size = 1
        hidden = hidden.view(model.num_layers, 2, batch_size, model.hidden_dim)
        cell = cell.view(model.num_layers, 2, batch_size, model.hidden_dim)
        hidden = torch.cat([hidden[:, 0, :, :], hidden[:, 1, :, :]], dim=2)
        cell = torch.cat([cell[:, 0, :, :], cell[:, 1, :, :]], dim=2)
        hidden = torch.tanh(model.bridge_h(hidden))
        cell = torch.tanh(model.bridge_c(cell))
        
        # Decode
        generated_ids = []
        input_token = torch.tensor([[word2idx["<sos>"]]], dtype=torch.long).to(DEVICE)
        
        for step in range(max_len):
            dec_input = model.embedding(input_token)
            
            # Attention
            hidden_repeated = hidden[-1].unsqueeze(1).repeat(1, enc_outputs.size(1), 1)
            attn_input = torch.cat([hidden_repeated, enc_outputs], dim=2)
            attn_weights = model.attention(attn_input).squeeze(2)
            attn_weights = F.softmax(attn_weights, dim=1)
            context = torch.bmm(attn_weights.unsqueeze(1), enc_outputs)
            rnn_input = torch.cat([dec_input, context], dim=2)
            
            output, (hidden, cell) = model.decoder(rnn_input, (hidden, cell))
            logits = model.fc(output.squeeze(1))
            
            # ⭐ Apply repetition penalty
            for prev_id in generated_ids[-5:]:  # Penalize last 5 tokens
                logits[0, prev_id] /= repetition_penalty
            
            next_id = logits.argmax(1).item()
            
            if next_id == word2idx["<eos>"]:
                break
            
            if next_id not in [word2idx["<unk>"], word2idx["<pad>"], word2idx["<sos>"]]:
                generated_ids.append(next_id)
            
            input_token = torch.tensor([[next_id]], dtype=torch.long).to(DEVICE)
        
        # Convert to text
        tokens = [idx2word[i] for i in generated_ids]
        return ''.join(tokens)

print("✓ Generation function ready (with repetition penalty)")

## TRAIN THE STRONGER MODEL!

In [ ]:
train_losses = []
val_losses = []
best_val_loss = float('inf')

print("\n" + "="*60)
print("TRAINING STRONGER MODEL")
print("="*60)
print(f"Layers: {NUM_LAYERS}, Hidden: {HIDDEN_DIM}, Dropout: {DROPOUT}")
print("="*60 + "\n")

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print("─" * 60)
    
    # Unfreeze embeddings at epoch 8
    if epoch == UNFREEZE_EPOCH:
        print("\n⭐ Unfreezing embeddings for fine-tuning...")
        model.embedding.weight.requires_grad = True
        for param_group in optimizer.param_groups:
            param_group['lr'] = 0.0005  # Lower LR for fine-tuning
        print(f"   New LR: {optimizer.param_groups[0]['lr']}\n")
    
    # Train
    train_loss = train_epoch(model, train_loader, criterion, optimizer, DEVICE, TEACHER_FORCING_RATIO)
    val_loss = validate(model, val_loader, criterion, DEVICE)
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    
    # Update LR
    scheduler.step(val_loss)
    
    # Results
    gap = val_loss - train_loss
    print(f"\nTrain: {train_loss:.4f} | Val: {val_loss:.4f} | Gap: {gap:.4f} | LR: {optimizer.param_groups[0]['lr']:.6f}")
    
    # Save best
    if val_loss < best_val_loss:
        improvement = best_val_loss - val_loss
        best_val_loss = val_loss
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': train_loss,
            'val_loss': val_loss,
        }, 'best_model_strong.pth')
        print(f"✓ BEST MODEL (improvement: {improvement:.4f})")
    
    # Test generation every 5 epochs
    if (epoch + 1) % 5 == 0:
        test_text = "မော်လ်တာကမ်းလွန်မှာ လှေမှောက် ရွှေ့ပြောင်းနေထိုင်သူ သေဆုံး"
        generated = generate_headline_greedy(model, test_text, max_len=15)
        print(f"\nTest generation: {generated}")
    
    # Early stopping
    if val_loss < 2.5:
        print("\n🎉 Reached target loss < 2.5!")
        break

print(f"\n\n" + "="*60)
print("TRAINING COMPLETE")
print("="*60)
print(f"Best val loss: {best_val_loss:.4f}")
print(f"Total epochs: {len(train_losses)}")
print("="*60)

## Plot Results

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(train_losses, 'o-', label='Train', linewidth=2)
plt.plot(val_losses, 's-', label='Val', linewidth=2)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.title('Training History (Stronger Model)', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
gaps = [v - t for t, v in zip(train_losses, val_losses)]
plt.plot(gaps, 'o-', color='red', linewidth=2)
plt.axhline(y=0.3, color='green', linestyle='--', alpha=0.5, label='Target gap < 0.3')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Val - Train Gap', fontsize=12)
plt.title('Overfitting Check', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nInitial → Final: {train_losses[0]:.4f} → {train_losses[-1]:.4f}")
print(f"Total improvement: {train_losses[0] - train_losses[-1]:.4f}")
print(f"Final gap: {gaps[-1]:.4f}")

## Test Final Model

In [ ]:
# Load best checkpoint
checkpoint = torch.load('best_model_strong.pth')
model.load_state_dict(checkpoint['model_state_dict'])
print(f"✓ Loaded best model (epoch {checkpoint['epoch']+1}, val loss {checkpoint['val_loss']:.4f})\n")

# Test on examples
test_examples = [
    "မော်လ်တာကမ်းလွန်မှာ လှေမှောက် ရွှေ့ပြောင်းနေထိုင်သူ သေဆုံး",
    "ရုရှား အတိုက်အခံ နိုင်ငံရေးသမား နာဗယ်လ်ညီ သေဆုံး",
]

print("="*60)
print("FINAL MODEL TEST")
print("="*60)

for i, text in enumerate(test_examples, 1):
    generated = generate_headline_greedy(model, text, max_len=20, repetition_penalty=1.5)
    print(f"\nExample {i}:")
    print(f"Article: {text}")
    print(f"Generated: {generated}")
    print("─" * 60)

# Test on validation examples
print("\nValidation Examples:")
for i in range(3):
    text_str = ''.join(val_texts[i][:100])
    actual = ''.join(val_headlines[i])
    generated = generate_headline_greedy(model, text_str, max_len=20)
    
    print(f"\n{i+1}.")
    print(f"   Actual:    {actual}")
    print(f"   Generated: {generated}")

## Save Everything

In [ ]:
# Save final model
torch.save(model.state_dict(), "seq2seq_strong_final.pth")

# Save vocab
with open("vocab_strong.pkl", "wb") as f:
    pickle.dump({"word2idx": word2idx, "idx2word": idx2word, "vocab_size": vocab_size}, f)

# Save history
with open("training_history_strong.pkl", "wb") as f:
    pickle.dump({
        "train_losses": train_losses,
        "val_losses": val_losses,
        "best_val_loss": best_val_loss,
        "config": {
            "hidden_dim": HIDDEN_DIM,
            "num_layers": NUM_LAYERS,
            "dropout": DROPOUT,
            "vocab_size": vocab_size,
        }
    }, f)

print("\n✓ All files saved!")
print("  - seq2seq_strong_final.pth")
print("  - best_model_strong.pth")
print("  - vocab_strong.pkl")
print("  - training_history_strong.pkl")